# Project Scenario:

In this project, you will put all the skills acquired throughout the course and your knowledge of basic Python to test. You will work on real-world data and perform the operations of Extraction, Transformation, and Loading as required. Throughout the project, you will note some outputs you need to answer questions on the graded quiz. You will also take snapshots, which you will upload in the AI-Graded or Peer-Graded Assignment, depending on the option you selected.  
Project Scenario

A multi-national firm has hired you as a data engineer. Your job is to access and process data as per requirements.

Your boss asked you to compile the list of the top 10 largest banks in the world ranked by market capitalization in billion USD. Further, you need to transform the data and store it in USD, GBP, EUR, and INR per the exchange rate information made available to you as a CSV file. You should save the processed information table locally in a CSV format and as a database table. Managers from different countries will query the database table to extract the list and note the market capitalization value in their own currency.

Directions

* Write a function to extract the tabular information from the given URL under the heading By Market Capitalization, and save it to a data frame.
* Write a function to transform the data frame by adding columns for Market Capitalization in GBP, EUR, and INR, rounded to 2 decimal places, based on the exchange rate information shared as a CSV file.
* Write a function to load the transformed data frame to an output CSV file.
* Write a function to load the transformed data frame to an SQL database server as a table.
* Write a function to run queries on the database table.
* Run the following queries on the database table: a. Extract the information for the London office, that is Name and MC_GBP_Billion b. Extract the information for the Berlin office, that is Name and MC_EUR_Billion c. Extract the information for New Delhi office, that is Name and MC_INR_Billion
* Write a function to log the progress of the code.
* While executing the data initialization commands and function calls, maintain appropriate log entries.



In [73]:
from bs4 import BeautifulSoup
import datetime as dt
import lxml
import numpy as np
import pandas as pd
import requests
import sqlite3

In [ ]:
# Code for ETL operations on Country-GDP data

def log_progress(message):
    ''' This function logs the mentioned message of a given stage of the
    code execution to a log file. Function returns nothing'''

    timestamp_format = '%Y-%h-%d-%H:%M:%S' # Year-Monthname-Day-Hour-Minute-Second 
    now = dt.datetime.now(tz=None) # get current timestamp 
    timestamp = now.strftime(timestamp_format)
    with open(log_file,"a") as f: 
        f.write(timestamp + ' : ' + message + '\n')

def extract(filename, table_attribs):
    ''' This function aims to extract the required
    information from the website and save it to a data frame. The
    function returns the data frame for further processing. 
    
    The original function as defined by the problem has url instead of
    filename, this is because I can't access archive.org, so I query
    a locally downloaded version of the website instead.'''
    df = pd.DataFrame(columns=table_attribs)

    with open("banks.html") as fp:
        data = BeautifulSoup(fp, "html.parser")

    table_target_position = 2
    bank_name_target_position = 0
    bank_name_text_target = 1
    bank_cap_target = 2

    tables = data.find_all("tbody")
    rows = tables[table_target_position].find_all("tr")

    for row in rows:
        td = row.find_all("td")
        if len(td) > 0:
            entry = {
                "Name": td[bank_name_target_position].find_all("a")[bank_name_text_target].text,
                "MC_USD_Billion": float(td[bank_cap_target].text)
            }
            df = pd.concat([df, pd.DataFrame(entry, index=[0])])
    return df

def transform(df, csv_path):
    ''' This function accesses the CSV file for exchange rate
	information, and adds three columns to the data frame, each
	containing the transformed version of Market Cap column to
	respective currencies'''

    exchange_rate = pd.read_csv(csv_path)

    gdp = round(df["MC_USD_Billion"] * exchange_rate.iloc[1,1], 2)
    eur = round(df["MC_USD_Billion"] * exchange_rate.iloc[0,1], 2)
    inr = round(df["MC_USD_Billion"] * exchange_rate.iloc[2,1], 2)

    df["MC_GDP_Billion"] = gdp
    df["MC_EUR_Billion"] = eur
    df["MC_INR_Billion"] = inr

    return df

def load_to_csv(df, output_path):
    ''' This function saves the final data frame as a CSV file in
	the provided path. Function returns nothing.'''
    df.to_csv(output_path, index=False)    

def load_to_db(df, sql_connection, table_name):
    ''' This function saves the final data frame to a database
	table with the provided name. Function returns nothing.'''
    df.to_sql(table_name, conn, if_exists='replace', index=False)

def run_query(query_statement, sql_connection):
    ''' This function runs the query on the database table and
    prints the output on the terminal. Function returns nothing. '''
    result = pd.read_sql(query_statement, sql_connection)
    print(result)

''' Here, you define the required entities and call the relevant
functions in the correct order to complete the project. Note that this
portion is not inside any function.'''

log_progress("Declaring known values")

#data_url = "https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks"
data_url = "banks.html"
#exchange_rate_csv_path = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-PY0221EN-Coursera/labs/v2/exchange_rate.csv"
exchange_rate_csv_path = "exchange_rate.csv"
table_attributes_extraction = ["Name", "MC_USD_Billion"]
table_attributes_final = ["Name", "MC_USD_Billion", "MC_GBP_Billion", "MC_EUR_Billion", "MC_INR_Billion"]
output_csv_path = "./Largest_banks_data.csv"
database_name = "Banks.db"
table_name = "Largest_banks"
log_file = "code_log.txt"

log_progress("Preliminaries complete. Initiating ETL process")

df = extract("banks.html",table_attributes_extraction)

log_progress("Data extraction complete. Initiating Transformation process")

df = transform(df, exchange_rate_csv_path)

log_progress("Data transformation complete. Initiating Loading process")

load_to_csv(df, output_csv_path)

log_progress("Data saved to CSV file")

conn = sqlite3.connect(database_name)

log_progress("SQL Connection initiated")

load_to_db(df, conn, table_name)

log_progress("Data loaded to Database as a table, Executing queries")

query_gdp = f"SELECT Name, MC_GDP_Billion from {table_name}"
query_eur = f"SELECT Name, MC_EUR_Billion from {table_name}"
query_inr = f"SELECT Name, MC_INR_Billion from {table_name}"

run_query(query_gdp, conn)
run_query(query_eur, conn)
run_query(query_inr, conn)

log_progress("Process Complete")

conn.close()

log_progress("Server Connection closed")

                                      Name  MC_GDP_Billion
0                           JPMorgan Chase          682.31
1                  China Construction Bank          331.26
2                          Bank of America          316.65
3               Agricultural Bank of China          281.64
4                           Morgan Stanley          271.83
5  Industrial and Commercial Bank of China          262.73
6                                     HSBC          252.74
7                            Goldman Sachs          250.06
8                            Bank of China          239.85
9                     Royal Bank of Canada          222.66
                                      Name  MC_EUR_Billion
0                           JPMorgan Chase          793.19
1                  China Construction Bank          385.09
2                          Bank of America          368.10
3               Agricultural Bank of China          327.41
4                           Morgan Stanley          316.